# Evaluator test: automatic labels vs human labels (150 SDXL images)

Compares the automatic labels of the main SDXL run with human labels on 150 held-out images (30 per complexity level 1–5), per constraint type.

- Automatic labels come from `labels.csv` of the complete SDXL run: 8-bit Qwen2.5-VL-7B for object, color (C), material (M) and second object (O2); OWLv2 for count (N) and position (P). For N and P, Qwen's own answers (`qwen_N`, `qwen_P`) are evaluated as well.
- Metrics per factor: accuracy with a 95% Wilson interval, Cohen's κ, the share of Yes labels for humans and for the evaluator, and both kinds of disagreement.
- Same rule as the pipeline: if the main object is absent, every constraint of the image fails. This rule is applied to the human labels as well.
- Human labels are normalised (case, spaces, typos such as `Falsa`). A missing N label is filled from the human object count in `notes` when available; other missing labels are excluded for that factor only.
- OWLv2 object counts are compared with the human counts, and strict prompt accuracy is compared per complexity level.
- A check against both runs confirms which model the reviewed images belong to.

No GPU is needed.

## 1. Inputs
Choose the human annotation file (`annotations_2_saida_maksim.csv`) in the dialog.

In [ ]:
from google.colab import drive, files
drive.mount("/content/drive")
import sys, json, importlib
from pathlib import Path
import pandas as pd
from IPython.display import display
SDXL_LABELS = Path("/content/drive/MyDrive/sdxl_v1/main_sdxl 2/labels.csv")
SD15_LABELS = Path("/content/drive/MyDrive/attribute_interference/v1/main_sd15/labels.csv")
OUT = Path("/content/drive/MyDrive/sdxl_v1/human_agreement_150")
TOOLS = Path("/content/agree_tools")
TOOLS.mkdir(exist_ok=True)
(TOOLS / "agree.py").write_text('import csv, math\nfrom pathlib import Path\n\nFACTORS = [\'object\', \'C\', \'N\', \'P\', \'M\', \'O2\']\nWORDS = {\'two\': 2, \'three\': 3, \'four\': 4}\n\n\ndef norm(v):\n    v = (v or \'\').strip().lower()\n    return {\'true\': True, \'false\': False, \'falsa\': False, \'ture\': True, \'flase\': False}.get(v)\n\n\ndef load_human(path):\n    text = Path(path).read_text(encoding=\'utf-8-sig\')\n    delim = \';\' if text.split(\'\\n\', 1)[0].count(\';\') > text.split(\'\\n\', 1)[0].count(\',\') else \',\'\n    rows = list(csv.DictReader(text.splitlines(), delimiter=delim))\n    out, notes = {}, []\n    for r in rows:\n        active = [f for f in r[\'active_factors\'].strip().split(\'|\') if f]\n        words = r[\'prompt\'].split()\n        requested = next((WORDS[w] for w in words if w in WORDS), None)\n        try:\n            count = int(r.get(\'notes\', \'\').strip())\n        except ValueError:\n            count = None\n        lab = {\'object\': norm(r[\'human_object\'])}\n        for f in active:\n            lab[f] = norm(r[f\'human_{f}\'])\n            if f == \'N\' and lab[f] is None and count is not None:\n                lab[f] = count == requested\n                notes.append(f"{r[\'image_id\']}: N filled from count {count} (requested {requested})")\n        if lab[\'object\'] is False:\n            for f in active:\n                lab[f] = False\n        for f in active:\n            if lab[f] is None:\n                notes.append(f"{r[\'image_id\']}: no human label for {f}, excluded for this factor")\n        out[r[\'image_id\'].strip()] = {\'labels\': lab, \'active\': active, \'count\': count, \'requested\': requested,\n                                      \'complexity\': len(active), \'prompt\': r[\'prompt\'].strip(),\n                                      \'reviewer\': r.get(\'reviewer\', \'\').strip()}\n    return out, notes\n\n\ndef load_auto(path):\n    rows = list(csv.DictReader(open(path, newline=\'\', encoding=\'utf-8\')))\n    out = {}\n    for r in rows:\n        obj = norm(r[\'object\'])\n        auto = {\'pipeline\': {}, \'qwen_only\': {}}\n        for f in FACTORS:\n            auto[\'pipeline\'][f] = norm(r[f])\n            q = norm(r[\'qwen_object\']) if f == \'object\' else norm(r[f\'qwen_{f}\'])\n            if f != \'object\' and r[f\'qwen_{f}\'] != \'\' and obj is False:\n                q = False\n            auto[\'qwen_only\'][f] = q\n        auto[\'owl_count\'] = int(float(r[\'owl_count\'])) if r.get(\'owl_count\') not in (None, \'\') else None\n        out[r[\'image_id\']] = auto\n    return out\n\n\ndef wilson(k, n, z=1.96):\n    if n == 0:\n        return None, None\n    p = k / n\n    d = 1 + z * z / n\n    c = (p + z * z / (2 * n)) / d\n    h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d\n    return max(0.0, c - h), min(1.0, c + h)\n\n\ndef kappa(pairs):\n    n = len(pairs)\n    if n == 0:\n        return None\n    po = sum(a == b for a, b in pairs) / n\n    p1 = sum(a for a, _ in pairs) / n\n    p2 = sum(b for _, b in pairs) / n\n    pe = p1 * p2 + (1 - p1) * (1 - p2)\n    return (po - pe) / (1 - pe) if pe < 1 else None\n\n\ndef compare(human, auto, evaluator, factor):\n    pairs = []\n    for iid, h in human.items():\n        if iid not in auto or (factor != \'object\' and factor not in h[\'active\']):\n            continue\n        a, b = h[\'labels\'].get(factor), auto[iid][evaluator].get(factor)\n        if a is not None and b is not None:\n            pairs.append((a, b))\n    n = len(pairs)\n    correct = sum(a == b for a, b in pairs)\n    lo, hi = wilson(correct, n)\n    k = kappa(pairs)\n    return {\'factor\': factor, \'evaluator\': evaluator, \'n\': n,\n            \'accuracy_pct\': round(100 * correct / n, 1) if n else None,\n            \'ci95_pct\': f\'{100 * lo:.0f}-{100 * hi:.0f}\' if n else None,\n            \'kappa\': round(k, 3) if k is not None else None,\n            \'human_yes_pct\': round(100 * sum(a for a, _ in pairs) / n, 1) if n else None,\n            \'auto_yes_pct\': round(100 * sum(b for _, b in pairs) / n, 1) if n else None,\n            \'auto_yes_human_no\': sum(b and not a for a, b in pairs),\n            \'auto_no_human_yes\': sum(a and not b for a, b in pairs)}\n\n\ndef per_factor(human, auto):\n    rows = []\n    for f in FACTORS:\n        rows.append({**compare(human, auto, \'pipeline\', f),\n                     \'evaluator\': \'OWLv2\' if f in {\'N\', \'P\'} else \'Qwen\'})\n        if f in {\'N\', \'P\'}:\n            rows.append({**compare(human, auto, \'qwen_only\', f), \'evaluator\': \'Qwen\'})\n    return rows\n\n\ndef counts(human, auto):\n    pairs = [(h[\'count\'], auto[i][\'owl_count\']) for i, h in human.items()\n             if i in auto and h[\'count\'] is not None and auto[i][\'owl_count\'] is not None and h[\'labels\'].get(\'object\') is True]\n    n = len(pairs)\n    if not n:\n        return {\'images\': 0}\n    errs = [b - a for a, b in pairs]\n    return {\'images\': n, \'exact_pct\': round(100 * sum(e == 0 for e in errs) / n, 1),\n            \'mae\': round(sum(abs(e) for e in errs) / n, 2), \'mean_signed_err\': round(sum(errs) / n, 2),\n            \'owl_over\': sum(e > 0 for e in errs), \'owl_under\': sum(e < 0 for e in errs)}\n\n\ndef strict(human, auto):\n    rows = []\n    for c in list(range(1, 6)) + [\'all\']:\n        hv, av, both = [], [], []\n        for i, h in human.items():\n            if i not in auto or (c != \'all\' and h[\'complexity\'] != c):\n                continue\n            fs = [\'object\'] + h[\'active\']\n            hl = [h[\'labels\'].get(f) for f in fs]\n            al = [auto[i][\'pipeline\'].get(f) for f in fs]\n            if None in hl or None in al:\n                continue\n            hv.append(all(hl)); av.append(all(al)); both.append(all(hl) == all(al))\n        n = len(hv)\n        rows.append({\'complexity\': c, \'images\': n,\n                     \'human_strict_pct\': round(100 * sum(hv) / n, 1) if n else None,\n                     \'auto_strict_pct\': round(100 * sum(av) / n, 1) if n else None,\n                     \'agreement_pct\': round(100 * sum(both) / n, 1) if n else None})\n    return rows\n\n\ndef model_check(human, autos):\n    rows = []\n    for name, auto in autos.items():\n        pairs = []\n        for f in [\'object\', \'C\', \'M\', \'O2\']:\n            for i, h in human.items():\n                if i in auto and (f == \'object\' or f in h[\'active\']):\n                    a, b = h[\'labels\'].get(f), auto[i][\'pipeline\'].get(f)\n                    if a is not None and b is not None:\n                        pairs.append(a == b)\n        rows.append({\'run\': name, \'matched_ids\': sum(i in auto for i in human),\n                     \'agreement_on_object_C_M_O2_pct\': round(100 * sum(pairs) / len(pairs), 1) if pairs else None})\n    return rows\n')
sys.path.insert(0, str(TOOLS))
import agree
importlib.reload(agree)
uploaded = files.upload()
name = next(n for n in uploaded if n.endswith(".csv"))
HUMAN = TOOLS / "human_annotations.csv"
HUMAN.write_bytes(uploaded[name])
human, notes = agree.load_human(HUMAN)
print(len(human), "reviewed images | reviewers:", sorted({h["reviewer"] for h in human.values()}))
print("by complexity:", pd.Series([h["complexity"] for h in human.values()]).value_counts().sort_index().to_dict())
print("\n".join(notes))

Mounted at /content/drive


Saving annotations_2_saida_maksim.csv to annotations_2_saida_maksim.csv
150 reviewed images | reviewers: ['Saida']
by complexity: {1: 30, 2: 30, 3: 30, 4: 30, 5: 30}
0a4a9df7d8b768b6_s2: N filled from count 2 (requested 4)
e9a6dac8f4829c57_s1: no human label for P, excluded for this factor
2ea37da8dd959a72_s0: N filled from count 4 (requested 3)


In [ ]:
import tarfile
TAR = Path("/content/drive/MyDrive/main_sdxl_full_20261027.tar")
with tarfile.open(TAR) as t:
    m = next(x for x in t if x.isfile() and x.name.rstrip("/").split("/")[-1] == "labels.csv" and "analysis" not in x.name)
    print("found in archive:", m.name)
    Path("/content/sdxl_labels.csv").write_bytes(t.extractfile(m).read())
lab = pd.read_csv("/content/sdxl_labels.csv")
print("labels rows:", len(lab), "| qwen ok:", (lab.qwen_status == "ok").sum(), "| owl ok:", (lab.owl_status == "ok").sum())

found in archive: main_sdxl/labels.csv
labels rows: 2304 | qwen ok: 2304 | owl ok: 1728


In [ ]:
SDXL_LABELS = Path("/content/sdxl_labels.csv")

## 2. Which run do the reviewed images belong to?
Image ids are shared by both runs, so agreement on object, C, M and O2 is checked against each. The reviewed run should agree clearly better.

In [ ]:
autos = {"SDXL": agree.load_auto(SDXL_LABELS)}
if SD15_LABELS.exists():
    autos["SD 1.5"] = agree.load_auto(SD15_LABELS)
display(pd.DataFrame(agree.model_check(human, autos)))
auto = autos["SDXL"]

,run,matched_ids,agreement_on_object_C_M_O2_pct
0,SDXL,150,85.2
1,SD 1.5,150,64.8


## 3. Accuracy and Cohen's κ per factor

In [ ]:
factor_table = pd.DataFrame(agree.per_factor(human, auto))
display(factor_table)

,factor,evaluator,n,accuracy_pct,ci95_pct,kappa,human_yes_pct,auto_yes_pct,auto_yes_human_no,auto_no_human_yes
0,object,Qwen,150,96.0,92-98,0.551,94.7,96.0,4,2
1,C,Qwen,90,78.9,69-86,0.568,67.8,53.3,3,16
2,N,OWLv2,90,77.8,68-85,0.472,37.8,17.8,1,19
3,N,Qwen,90,87.8,79-93,0.742,37.8,38.9,6,5
4,P,OWLv2,89,88.8,81-94,0.237,4.5,11.2,8,2
5,P,Qwen,89,94.4,88-98,0.415,4.5,5.6,3,2
6,M,Qwen,90,71.1,61-79,0.448,71.1,44.4,1,25
7,O2,Qwen,90,87.8,79-93,0.756,48.9,56.7,9,2


## 4. OWLv2 object counts vs human counts
Images with a count or position factor where the object is present.

In [ ]:
count_summary = agree.counts(human, auto)
print(json.dumps(count_summary, indent=2))

{
  "images": 114,
  "exact_pct": 40.4,
  "mae": 2.09,
  "mean_signed_err": 1.44,
  "owl_over": 54,
  "owl_under": 14
}


## 5. Strict prompt accuracy: humans vs pipeline

In [ ]:
strict_table = pd.DataFrame(agree.strict(human, auto))
display(strict_table)

,complexity,images,human_strict_pct,auto_strict_pct,agreement_pct
0,1,30,66.7,70.0,83.3
1,2,30,26.7,30.0,83.3
2,3,29,10.3,0.0,89.7
3,4,30,0.0,0.0,100.0
4,5,30,0.0,0.0,100.0
5,all,149,20.8,20.1,91.3


## 6. Disagreements to inspect

In [ ]:
rows = []
for iid, h in human.items():
    for f in ["object"] + h["active"]:
        a, b = h["labels"].get(f), auto[iid]["pipeline"].get(f) if iid in auto else None
        if a is not None and b is not None and a != b:
            rows.append({"image_id": iid, "factor": f, "human": a, "auto": b, "prompt": h["prompt"]})
disagreements = pd.DataFrame(rows)
print(len(disagreements), "disagreements")
display(disagreements.sort_values(["factor", "image_id"]).head(60))

92 disagreements


,image_id,factor,human,auto,prompt
5,0a4a9df7d8b768b6_s2,C,False,True,a photo of four black tables on the left side ...
8,1d1af47990f8ff6e_s1,C,True,False,a photo of a yellow tray and a green plate
29,227631371a4b3a3d_s3,C,True,False,a photo of a blue glass box on the right side ...
0,237f4075229b3a82_s1,C,True,False,a photo of a green glass stool and a blue book
64,31b4d61ee9680166_s3,C,False,True,a photo of a blue dog and a yellow plate
39,3bdba2057eba7cdc_s0,C,True,False,a photo of four red glass clocks on the right ...
70,457653c2784f4715_s1,C,True,False,a photo of a black wooden vase on the right si...
2,56d19a69d71ad1d6_s2,C,True,False,a photo of a blue bench on the left side of th...
75,5cf8dc60e827871b_s1,C,True,False,a photo of three green glass chairs on the lef...
86,5cf8dc60e827871b_s3,C,True,False,a photo of three green glass chairs on the lef...


In [ ]:
OUT = Path("/content/drive/MyDrive/human_agreement_150")

## 7. Save

In [ ]:
OUT.mkdir(exist_ok=True)
factor_table.to_csv(OUT / "agreement_per_factor.csv", index=False)
strict_table.to_csv(OUT / "strict_human_vs_auto.csv", index=False)
disagreements.to_csv(OUT / "disagreements.csv", index=False)
(OUT / "owl_count_vs_human.json").write_text(json.dumps(count_summary, indent=2))
(OUT / "human_annotations.csv").write_bytes(HUMAN.read_bytes())
(OUT / "notes.json").write_text(json.dumps({"label_fixes": notes, "run": "main_sdxl (complete, 2304 images)"}, indent=2))
print("Saved to", OUT, sorted(p.name for p in OUT.iterdir()))

Saved to /content/drive/MyDrive/human_agreement_150 ['agreement_per_factor.csv', 'disagreements.csv', 'human_annotations.csv', 'notes.json', 'owl_count_vs_human.json', 'strict_human_vs_auto.csv']
